# MM60 — Mestre de Materiais

**Tabela:** `dev_procurement.corp_curated.tbl_ds_mdm_mm60`
**Transação SAP:** MM60 · **Colunas:** 17
**Clustering declarado:** `cod_material`, `cod_centro`

---

## Objetivo
Mapear o comportamento desta tabela **antes** de qualquer comparação com o SAP.
O resultado alimenta a base de conhecimento do agente de validação e define o cenário de teste.

## Como usar
1. Execute a célula **1** para criar os widgets, depois ajuste os filtros no topo (opcional).
2. Execute a célula **2** — ela cria a view `base`, usada por todas as demais.
3. Execute as células na ordem e leia a coluna **`veredito`** de cada resultado.
4. Exporte o notebook executado para a pasta de conhecimento do agente.

## Aviso metodológico
Contagem de linhas **não** é evidência de qualidade. Erros de colapso de granularidade
preservam o total. Ver seções **4**, **5** e **14**.


## 1. Widgets de recorte

Execute uma vez. Deixe vazio para analisar a base completa.

In [0]:
-- 1. WIDGETS (deixe vazio = sem filtro)
-- Parametros definidos como widgets do notebook (use os campos no topo da pagina).
SELECT
  :f_cod_centro AS f_cod_centro,
  :f_tp_material AS f_tp_material,
  :f_cod_grupo_mercadoria AS f_cod_grupo_mercadoria;

## 2. View `base`

Aplica os filtros dos widgets uma única vez. **Todas** as células seguintes consultam `base`.

In [0]:
-- 2. VIEW BASE (aplica os filtros dos widgets)
CREATE OR REPLACE TEMP VIEW base AS
SELECT * FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60
WHERE (:f_cod_centro = '' OR `cod_centro` = :f_cod_centro)
  AND (:f_tp_material = '' OR `tp_material` = :f_tp_material)
  AND (:f_cod_grupo_mercadoria = '' OR `cod_grupo_mercadoria` = :f_cod_grupo_mercadoria);

SELECT COUNT(*) AS linhas_na_base FROM base;

## 3. Metadados e histórico de carga

Formato, particionamento, clustering real e última atualização.
Divergência entre clustering declarado e chave real é o primeiro indício de problema.

In [0]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_mdm_mm60;

In [0]:
-- Formato, tamanho e particoes (falha se nao for Delta)
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_mdm_mm60;

In [0]:
-- Ultimas operacoes de escrita (falha se for view ou nao-Delta)
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_mdm_mm60 LIMIT 20;

## 4. Granularidade real

`linhas ÷ chaves distintas`. Razão maior que 1,00 significa que existe uma dimensão
adicional multiplicando as linhas — é preciso descobrir **qual** (seção 14).

In [0]:
-- 4. GRANULARIDADE REAL: linhas / chaves distintas
-- Razao > 1,00 significa que existe dimensao adicional multiplicando linhas.
WITH t AS (SELECT COUNT(*) AS total FROM base),
g AS (
SELECT 'cod_material + cod_centro' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `cod_material`, `cod_centro` FROM base)
UNION ALL
SELECT 'cod_material + cod_centro + tp_avaliacao' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `tp_avaliacao` FROM base)
UNION ALL
SELECT 'cod_material' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `cod_material` FROM base)
)
SELECT g.chave, t.total AS linhas, g.combinacoes_distintas,
       ROUND(t.total / g.combinacoes_distintas, 4) AS linhas_por_chave,
       CASE WHEN g.combinacoes_distintas = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade por chave candidata

Quantas combinações se repetem e qual o pior caso.

In [0]:
-- 5. DUPLICIDADE POR CHAVE
SELECT 'cod_material + cod_centro' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `cod_material`, `cod_centro`, COUNT(*) AS qtd FROM base GROUP BY `cod_material`, `cod_centro` HAVING COUNT(*) > 1)
UNION ALL
SELECT 'cod_material + cod_centro + tp_avaliacao' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `cod_material`, `cod_centro`, `tp_avaliacao`, COUNT(*) AS qtd FROM base GROUP BY `cod_material`, `cod_centro`, `tp_avaliacao` HAVING COUNT(*) > 1)
UNION ALL
SELECT 'cod_material' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `cod_material`, COUNT(*) AS qtd FROM base GROUP BY `cod_material` HAVING COUNT(*) > 1)
ORDER BY chaves_repetidas DESC;

## 6. Varredura de preenchimento — TODAS as colunas

**Seção mais importante do notebook.**

Detecta coluna nunca carregada. Em validação anterior, esta análise revelou 7 colunas
100% nulas no Datalake — uma delas preenchida em **97,9%** dos registros do SAP.
Este erro **não aparece** em teste por amostragem.

Ordene pela coluna `veredito`: os problemas aparecem primeiro.

In [0]:
-- 6. PREENCHIMENTO DE TODAS AS COLUNAS
-- Detecta coluna nunca carregada. Secao mais importante do notebook.
WITH t AS (SELECT COUNT(*) AS total FROM base),
perf AS (
  SELECT stack(17,
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'tp_avaliacao', 'string', COUNT_IF(`tp_avaliacao` IS NULL), COUNT_IF(`tp_avaliacao` IS NOT NULL AND lower(trim(`tp_avaliacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_avaliacao`) RLIKE '^0+([.,]0+)?$'),
    'desc_material', 'string', COUNT_IF(`desc_material` IS NULL), COUNT_IF(`desc_material` IS NOT NULL AND lower(trim(`desc_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_material`) RLIKE '^0+([.,]0+)?$'),
    'sg_um_basica', 'string', COUNT_IF(`sg_um_basica` IS NULL), COUNT_IF(`sg_um_basica` IS NOT NULL AND lower(trim(`sg_um_basica`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_um_basica`) RLIKE '^0+([.,]0+)?$'),
    'tp_material', 'string', COUNT_IF(`tp_material` IS NULL), COUNT_IF(`tp_material` IS NOT NULL AND lower(trim(`tp_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_grupo_comprador', 'string', COUNT_IF(`cod_grupo_comprador` IS NULL), COUNT_IF(`cod_grupo_comprador` IS NOT NULL AND lower(trim(`cod_grupo_comprador`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_grupo_comprador`) RLIKE '^0+([.,]0+)?$'),
    'cod_grupo_mercadoria', 'string', COUNT_IF(`cod_grupo_mercadoria` IS NULL), COUNT_IF(`cod_grupo_mercadoria` IS NOT NULL AND lower(trim(`cod_grupo_mercadoria`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_grupo_mercadoria`) RLIKE '^0+([.,]0+)?$'),
    'nm_criado_por', 'string', COUNT_IF(`nm_criado_por` IS NULL), COUNT_IF(`nm_criado_por` IS NOT NULL AND lower(trim(`nm_criado_por`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_criado_por`) RLIKE '^0+([.,]0+)?$'),
    'vl_preco_brl', 'decimal(18,2)', COUNT_IF(`vl_preco_brl` IS NULL), 0L, COUNT_IF(`vl_preco_brl` = 0),
    'sg_moeda', 'string', COUNT_IF(`sg_moeda` IS NULL), COUNT_IF(`sg_moeda` IS NOT NULL AND lower(trim(`sg_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_moeda`) RLIKE '^0+([.,]0+)?$'),
    'dt_ultima_modificacao', 'string', COUNT_IF(`dt_ultima_modificacao` IS NULL), COUNT_IF(`dt_ultima_modificacao` IS NOT NULL AND lower(trim(`dt_ultima_modificacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^0+([.,]0+)?$'),
    'tp_mrp', 'string', COUNT_IF(`tp_mrp` IS NULL), COUNT_IF(`tp_mrp` IS NOT NULL AND lower(trim(`tp_mrp`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_mrp`) RLIKE '^0+([.,]0+)?$'),
    'cod_abc', 'string', COUNT_IF(`cod_abc` IS NULL), COUNT_IF(`cod_abc` IS NOT NULL AND lower(trim(`cod_abc`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_abc`) RLIKE '^0+([.,]0+)?$'),
    'cod_classe_avaliacao', 'string', COUNT_IF(`cod_classe_avaliacao` IS NULL), COUNT_IF(`cod_classe_avaliacao` IS NOT NULL AND lower(trim(`cod_classe_avaliacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_classe_avaliacao`) RLIKE '^0+([.,]0+)?$'),
    'tp_controle_preco', 'string', COUNT_IF(`tp_controle_preco` IS NULL), COUNT_IF(`tp_controle_preco` IS NOT NULL AND lower(trim(`tp_controle_preco`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_controle_preco`) RLIKE '^0+([.,]0+)?$'),
    'qt_unidade_preco', 'decimal(5,0)', COUNT_IF(`qt_unidade_preco` IS NULL), 0L, COUNT_IF(`qt_unidade_preco` = 0)
  ) AS (coluna, tipo, nulos, vazios, zeros)
  FROM base
)
SELECT
  p.coluna,
  p.tipo,
  p.nulos,
  p.vazios,
  p.zeros,
  t.total - p.nulos - p.vazios - p.zeros                              AS uteis,
  ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros) / t.total, 2) AS pct_util,
  CASE
    WHEN p.nulos = t.total                                      THEN '1. 100% NULO'
    WHEN t.total - p.nulos - p.vazios - p.zeros <= 0            THEN '2. SEM VALOR UTIL'
    WHEN (t.total - p.nulos - p.vazios - p.zeros) < t.total*0.01 THEN '3. QUASE VAZIO (<1%)'
    ELSE '9. ok'
  END AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade

Valores distintos por coluna. Coluna constante é candidata a default de carga.

In [0]:
-- 7. CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM base),
card AS (
  SELECT stack(17,
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'tp_avaliacao', 'string', approx_count_distinct(`tp_avaliacao`),
    'desc_material', 'string', approx_count_distinct(`desc_material`),
    'sg_um_basica', 'string', approx_count_distinct(`sg_um_basica`),
    'tp_material', 'string', approx_count_distinct(`tp_material`),
    'cod_grupo_comprador', 'string', approx_count_distinct(`cod_grupo_comprador`),
    'cod_grupo_mercadoria', 'string', approx_count_distinct(`cod_grupo_mercadoria`),
    'nm_criado_por', 'string', approx_count_distinct(`nm_criado_por`),
    'vl_preco_brl', 'decimal(18,2)', approx_count_distinct(`vl_preco_brl`),
    'sg_moeda', 'string', approx_count_distinct(`sg_moeda`),
    'dt_ultima_modificacao', 'string', approx_count_distinct(`dt_ultima_modificacao`),
    'tp_mrp', 'string', approx_count_distinct(`tp_mrp`),
    'cod_abc', 'string', approx_count_distinct(`cod_abc`),
    'cod_classe_avaliacao', 'string', approx_count_distinct(`cod_classe_avaliacao`),
    'tp_controle_preco', 'string', approx_count_distinct(`tp_controle_preco`),
    'qt_unidade_preco', 'decimal(5,0)', approx_count_distinct(`qt_unidade_preco`)
  ) AS (coluna, tipo, distintos)
  FROM base
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE
         WHEN c.distintos <= 1                   THEN '1. CONSTANTE (1 valor)'
         WHEN c.distintos <= 3                   THEN '2. cardinalidade muito baixa'
         WHEN c.distintos > t.total * 0.95       THEN '3. candidata a identificador'
         ELSE '9. normal'
       END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada coluna. Um valor concentrando mais de 99% da base
indica possível default de carga em vez de dado real.

In [0]:
-- 8. DOMINIO DAS COLUNAS CATEGORICAS (top 8 de cada)
-- Valor concentrando >99% indica possivel default de carga.
(SELECT 'cod_centro' AS coluna, CAST(`cod_centro` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_centro` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_avaliacao' AS coluna, CAST(`tp_avaliacao` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `tp_avaliacao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'sg_um_basica' AS coluna, CAST(`sg_um_basica` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `sg_um_basica` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_material' AS coluna, CAST(`tp_material` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `tp_material` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_grupo_comprador' AS coluna, CAST(`cod_grupo_comprador` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_grupo_comprador` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_grupo_mercadoria' AS coluna, CAST(`cod_grupo_mercadoria` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_grupo_mercadoria` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'sg_moeda' AS coluna, CAST(`sg_moeda` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `sg_moeda` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_mrp' AS coluna, CAST(`tp_mrp` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `tp_mrp` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_abc' AS coluna, CAST(`cod_abc` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_abc` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_classe_avaliacao' AS coluna, CAST(`cod_classe_avaliacao` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_classe_avaliacao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_controle_preco' AS coluna, CAST(`tp_controle_preco` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `tp_controle_preco` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 9. Perfil dos campos numéricos

**Atenção ao tipo:** quantidade costuma usar `decimal`, mas valor monetário
frequentemente usa `double` — risco de arredondamento na conciliação financeira.

In [0]:
-- 9. PERFIL DOS CAMPOS NUMERICOS
-- Tipo DOUBLE em valor monetario = risco de arredondamento na conciliacao.
SELECT * FROM (
  SELECT stack(2,
    'vl_preco_brl', 'decimal(18,2)', COUNT(`vl_preco_brl`), CAST(MIN(`vl_preco_brl`) AS DOUBLE), CAST(MAX(`vl_preco_brl`) AS DOUBLE), CAST(AVG(`vl_preco_brl`) AS DOUBLE), CAST(percentile_approx(`vl_preco_brl`, 0.5) AS DOUBLE), CAST(percentile_approx(`vl_preco_brl`, 0.95) AS DOUBLE), COUNT_IF(`vl_preco_brl` < 0), COUNT_IF(`vl_preco_brl` = 0),
    'qt_unidade_preco', 'decimal(5,0)', COUNT(`qt_unidade_preco`), CAST(MIN(`qt_unidade_preco`) AS DOUBLE), CAST(MAX(`qt_unidade_preco`) AS DOUBLE), CAST(AVG(`qt_unidade_preco`) AS DOUBLE), CAST(percentile_approx(`qt_unidade_preco`, 0.5) AS DOUBLE), CAST(percentile_approx(`qt_unidade_preco`, 0.95) AS DOUBLE), COUNT_IF(`qt_unidade_preco` < 0), COUNT_IF(`qt_unidade_preco` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, p95, negativos, zeros)
  FROM base
)
ORDER BY coluna;

## 10. Datas armazenadas como STRING

**Armadilha conhecida:** o SAP exporta `2024-02-23 00:00:00` e o Datalake grava `20240223`.
É a mesma data em formato diferente — já gerou **16.773 falsos positivos**.

Se a coluna `veredito` acusar mais de um formato, a normalização é obrigatória.

In [0]:
-- 10. DATAS ARMAZENADAS COMO STRING
-- ARMADILHA: SAP exporta '2024-02-23 00:00:00', Datalake grava '20240223'.
-- Mesma data, formato diferente. Ja gerou 16.773 falsos positivos.
WITH t AS (SELECT COUNT(*) AS total FROM base),
d AS (
  SELECT stack(1,
    'dt_ultima_modificacao', COUNT_IF(`dt_ultima_modificacao` IS NULL OR trim(`dt_ultima_modificacao`) = ''), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_ultima_modificacao`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_ultima_modificacao`) NOT IN ('', '00000000') THEN `dt_ultima_modificacao` END), MAX(CASE WHEN trim(`dt_ultima_modificacao`) NOT IN ('', '00000000') THEN `dt_ultima_modificacao` END)
  ) AS (coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, data_zero, minimo, maximo)
  FROM base
)
SELECT d.coluna, d.vazios, d.fmt_AAAAMMDD, d.fmt_ISO, d.fmt_BR, d.data_zero,
       t.total - d.vazios - d.fmt_AAAAMMDD - d.fmt_ISO - d.fmt_BR AS nao_reconhecido,
       d.minimo, d.maximo,
       CASE WHEN (CASE WHEN d.fmt_AAAAMMDD > 0 THEN 1 ELSE 0 END
                + CASE WHEN d.fmt_ISO       > 0 THEN 1 ELSE 0 END
                + CASE WHEN d.fmt_BR        > 0 THEN 1 ELSE 0 END) > 1
            THEN 'ALERTA: mais de um formato na mesma coluna'
            ELSE 'formato unico' END AS veredito
FROM d CROSS JOIN t
ORDER BY d.coluna;

## 11. Códigos — zeros à esquerda, espaços e formato

**Armadilha conhecida:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá **0% de match**.

A coluna `alertas` resume o que exige tratamento antes da comparação.

In [0]:
-- 11. CODIGOS: ZEROS A ESQUERDA, ESPACOS E FORMATO
-- ARMADILHA: SAP grava '425263', Datalake grava '000000000000425263'.
-- Sem normalizar, o join da 0% de match.
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq, com_espacos,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes_ao_remover_zeros,
       CONCAT_WS(' | ',
         CASE WHEN com_zeros_esq > 0 THEN 'tem zeros a esquerda' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN com_espacos > 0 THEN 'tem espacos' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END,
         CASE WHEN tipo LIKE '%int%' OR tipo LIKE 'big%'
              THEN 'TIPO NUMERICO - zeros a esquerda JA perdidos' END
       ) AS alertas
FROM (
  SELECT stack(3,
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_material` AS STRING) <> trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_centro', 'string', COUNT_IF(CAST(`cod_centro` AS STRING) IS NULL OR trim(CAST(`cod_centro` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro` AS STRING)))), MAX(length(trim(CAST(`cod_centro` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_centro` AS STRING) <> trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '')),
    'tp_avaliacao', 'string', COUNT_IF(CAST(`tp_avaliacao` AS STRING) IS NULL OR trim(CAST(`tp_avaliacao` AS STRING)) = ''), MIN(length(trim(CAST(`tp_avaliacao` AS STRING)))), MAX(length(trim(CAST(`tp_avaliacao` AS STRING)))), COUNT_IF(trim(CAST(`tp_avaliacao` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`tp_avaliacao` AS STRING) <> trim(CAST(`tp_avaliacao` AS STRING))), COUNT(DISTINCT trim(CAST(`tp_avaliacao` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`tp_avaliacao` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq, com_espacos,
        distintos_bruto, distintos_sem_zeros)
  FROM base
)
ORDER BY coluna;

## 12. Amostra de linhas completas

O dado como ele realmente está: formato de código, decimais, datas e nulos.

In [0]:
-- 12. AMOSTRA
SELECT * FROM base LIMIT 20;

In [0]:
-- 12.1 AMOSTRA ALEATORIA
SELECT * FROM base ORDER BY rand() LIMIT 10;

## 13. Distribuição por dimensão de recorte

Base para escolher o cenário de teste: volume viável (10 mil a 300 mil linhas)
contendo os casos-limite identificados nas seções anteriores.

In [0]:
-- 13. DISTRIBUICAO POR cod_centro
SELECT `cod_centro`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `cod_centro`
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 13. DISTRIBUICAO POR tp_material
SELECT `tp_material`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `tp_material`
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 13. DISTRIBUICAO POR cod_grupo_mercadoria
SELECT `cod_grupo_mercadoria`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `cod_grupo_mercadoria`
ORDER BY linhas DESC
LIMIT 40;

## 14. Duplicidade — o que diferencia as linhas repetidas?

Analisando pela chave **cod_material + cod_centro**.

**Regra crítica:** linhas **idênticas** = duplicata real (erro de carga).
Linhas **distintas** = granularidade adicional legítima (split valuation, lote, tipo de avaliação).

São problemas diferentes com tratamentos diferentes. Em validação anterior, 5 linhas do mesmo
material eram todas distintas, diferenciadas por um campo que sequer existia nos extratos do SAP.

In [0]:
-- 14. CHAVES DUPLICADAS
SELECT `cod_material`, `cod_centro`, COUNT(*) AS qtd
FROM base
GROUP BY `cod_material`, `cod_centro`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 20;

In [0]:
-- 14.1 O QUE DIFERENCIA AS LINHAS DUPLICADAS?
-- REGRA: linhas identicas = duplicata real (erro de carga).
--        linhas distintas = granularidade adicional legitima (split valuation, lote...).
WITH dup AS (
  SELECT `cod_material`, `cod_centro` FROM base GROUP BY `cod_material`, `cod_centro` HAVING COUNT(*) > 1
),
d AS (
  SELECT b.* FROM base b JOIN dup USING (`cod_material`, `cod_centro`)
),
agg AS (
  SELECT `cod_material`, `cod_centro`,
         COUNT(DISTINCT `tp_avaliacao`) AS `tp_avaliacao`,
         COUNT(DISTINCT `desc_material`) AS `desc_material`,
         COUNT(DISTINCT `sg_um_basica`) AS `sg_um_basica`,
         COUNT(DISTINCT `tp_material`) AS `tp_material`,
         COUNT(DISTINCT `cod_grupo_comprador`) AS `cod_grupo_comprador`,
         COUNT(DISTINCT `cod_grupo_mercadoria`) AS `cod_grupo_mercadoria`,
         COUNT(DISTINCT `nm_criado_por`) AS `nm_criado_por`,
         COUNT(DISTINCT `vl_preco_brl`) AS `vl_preco_brl`,
         COUNT(DISTINCT `sg_moeda`) AS `sg_moeda`,
         COUNT(DISTINCT `dt_ultima_modificacao`) AS `dt_ultima_modificacao`,
         COUNT(DISTINCT `tp_mrp`) AS `tp_mrp`,
         COUNT(DISTINCT `cod_abc`) AS `cod_abc`,
         COUNT(DISTINCT `cod_classe_avaliacao`) AS `cod_classe_avaliacao`,
         COUNT(DISTINCT `tp_controle_preco`) AS `tp_controle_preco`,
         COUNT(DISTINCT `qt_unidade_preco`) AS `qt_unidade_preco`
  FROM d GROUP BY `cod_material`, `cod_centro`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1
            THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(15,
    'tp_avaliacao', MAX(`tp_avaliacao`),
    'desc_material', MAX(`desc_material`),
    'sg_um_basica', MAX(`sg_um_basica`),
    'tp_material', MAX(`tp_material`),
    'cod_grupo_comprador', MAX(`cod_grupo_comprador`),
    'cod_grupo_mercadoria', MAX(`cod_grupo_mercadoria`),
    'nm_criado_por', MAX(`nm_criado_por`),
    'vl_preco_brl', MAX(`vl_preco_brl`),
    'sg_moeda', MAX(`sg_moeda`),
    'dt_ultima_modificacao', MAX(`dt_ultima_modificacao`),
    'tp_mrp', MAX(`tp_mrp`),
    'cod_abc', MAX(`cod_abc`),
    'cod_classe_avaliacao', MAX(`cod_classe_avaliacao`),
    'tp_controle_preco', MAX(`tp_controle_preco`),
    'qt_unidade_preco', MAX(`qt_unidade_preco`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 15. Freshness — atualidade da carga

In [0]:
-- 15. FRESHNESS
-- Esta tabela NAO possui coluna de data de ingestao.
-- ACAO: solicitar ao time de dados a inclusao de dateingest ou equivalente.
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_mdm_mm60 LIMIT 10;

## 16. Análises específicas — MM60

### 16.1 Split valuation — `tp_avaliacao`

O clustering declarado é **material + centro**, mas a tabela possui `tp_avaliacao`.
Se um material tem mais de um tipo de avaliação, a chave material+centro **deixa de ser única**.

Foi exatamente este o padrão que gerou 5 linhas para o mesmo material em validação anterior.

In [0]:
-- 16.1 SPLIT VALUATION
WITH g AS (
  SELECT cod_material, cod_centro,
         COUNT(DISTINCT tp_avaliacao) AS qt_avaliacoes,
         COUNT(*) AS linhas
  FROM base GROUP BY cod_material, cod_centro
)
SELECT qt_avaliacoes,
       COUNT(*) AS combinacoes_material_centro,
       SUM(linhas - 1) AS linhas_excedentes,
       CASE WHEN qt_avaliacoes > 1
            THEN 'CHAVE REAL inclui tp_avaliacao' ELSE 'material+centro suficiente' END AS veredito
FROM g GROUP BY qt_avaliacoes ORDER BY qt_avaliacoes;

In [0]:
-- 16.1b MATERIAIS COM SPLIT VALUATION
SELECT cod_material, cod_centro,
       COUNT(DISTINCT tp_avaliacao) AS qt_avaliacoes,
       CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(tp_avaliacao))) AS tipos
FROM base
GROUP BY cod_material, cod_centro
HAVING COUNT(DISTINCT tp_avaliacao) > 1
ORDER BY qt_avaliacoes DESC
LIMIT 25;

In [0]:
-- 16.1c FORMATO DO tp_avaliacao (zeros a esquerda inconsistentes?)
SELECT length(trim(tp_avaliacao)) AS comprimento,
       COUNT(*) AS linhas,
       COUNT(DISTINCT tp_avaliacao) AS valores_distintos
FROM base
WHERE tp_avaliacao IS NOT NULL AND trim(tp_avaliacao) <> ''
GROUP BY length(trim(tp_avaliacao))
ORDER BY comprimento;

### 16.2 Perfil de preço
`vl_preco_brl` é `decimal(18,2)` — adequado para conciliação financeira,
ao contrário dos campos `double` de outras tabelas.

In [0]:
-- 16.2 PRECO POR TIPO DE CONTROLE
SELECT tp_controle_preco,
       COUNT(*) AS linhas,
       COUNT_IF(vl_preco_brl IS NULL OR vl_preco_brl = 0) AS sem_preco,
       ROUND(AVG(vl_preco_brl), 2) AS preco_medio,
       ROUND(percentile_approx(vl_preco_brl, 0.5), 2) AS mediana,
       ROUND(MAX(vl_preco_brl), 2) AS maximo
FROM base
GROUP BY tp_controle_preco
ORDER BY linhas DESC;

## 90. Integridade referencial cruzada _(opcional)_

Confere se os códigos desta tabela existem nas tabelas de referência.
Execute apenas se as outras tabelas estiverem acessíveis no mesmo ambiente.

In [0]:
-- Sem verificacao de integridade configurada para esta tabela.
SELECT 'n/a' AS status;

## 99. Resumo consolidado

**Copie a saída desta célula** para o relatório ou para a base de conhecimento do agente.

In [0]:
-- 99. RESUMO CONSOLIDADO
SELECT 'VOLUMETRIA' AS bloco, 'linhas na base' AS item,
       CAST(COUNT(*) AS STRING) AS valor, '' AS veredito
  FROM base
UNION ALL
SELECT 'VOLUMETRIA', 'colunas', '17', ''
UNION ALL
SELECT 'VOLUMETRIA', 'clustering declarado',
       'cod_material, cod_centro', ''
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'cod_material + cod_centro' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `cod_material`, `cod_centro` FROM base)
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'cod_material + cod_centro + tp_avaliacao' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `tp_avaliacao` FROM base)
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'cod_material' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `cod_material` FROM base)
ORDER BY bloco, item;

---

## Próximo passo

1. Escolher o recorte de teste com base na **seção 13**.
2. Extrair a transação no SAP com o **mesmo recorte** e na **mesma data** do snapshot.
3. Extrair **todas** as abas/telas da transação — comparar parcialmente esconde erros de granularidade.
4. Submeter os arquivos ao agente de validação junto com este notebook executado.

### Checklist antes de comparar com o SAP

- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP antes de classificar como erro (seção 6)
- [ ] Zeros à esquerda normalizados nos dois lados (seção 11)
- [ ] Formato de data normalizado para `AAAAMMDD` (seção 10)
- [ ] Tolerância de 0,005 aplicada em campos `double` (seção 9)
- [ ] Duplicidades classificadas: idênticas vs granularidade legítima (seção 14)
